<img src='https://kayfa.ai/logo.png' width='200'>

# Employee Attrition Analytics
### Kayfa AI & Data Analytics Internship · Week 1 Task
---
*End-to-end analytics pipeline: load, clean, explore, and surface HR insights from 74,498 synthetic employee records.*

**Goal:** This notebook performs a deep-dive exploratory data analysis (EDA) to understand the drivers of employee attrition and provide data-backed recommendations for HR leadership.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

# Global visual config
COLORS   = {"Left": "#E45756", "Stayed": "#4E79A7"}
TEMPLATE = "plotly_white"

# Human-readable labels for charts (to avoid snake_case)
LABELS = {
    "attrition": "Attrition Status",
    "attrition_flag": "Attrition Rate (%)",
    "age": "Age",
    "age_group": "Age Group",
    "gender": "Gender",
    "years_at_company": "Years at Company",
    "tenure_stage": "Tenure Stage",
    "job_role": "Job Role",
    "monthly_income": "Monthly Income",
    "pay_band": "Pay Band",
    "work_life_balance": "Work-Life Balance",
    "job_satisfaction": "Job Satisfaction",
    "number_of_promotions": "Number of Promotions",
    "promotion_group": "Promotion Status",
    "overtime": "Overtime",
    "distance_from_home": "Distance from Home",
    "marital_status": "Marital Status",
    "number_of_dependents": "Number of Dependents",
    "dependents_group": "Dependents Group",
    "job_level": "Job Level",
    "remote_work": "Remote Work",
    "leadership_opportunities": "Leadership Opportunities",
    "innovation_opportunities": "Innovation Opportunities",
    "growth_access": "Growth Access",
    "stuck_profile": "Growth Profile",
    "total": "Total Employees",
    "left": "Employees Who Left",
    "rate": "Attrition Rate (%)"
}

## 1 · Load & Combine
We bring together `train.csv` and `test.csv` to ensure our analysis reflects the entire employee population.

In [2]:
try:
    train = pd.read_csv("train.csv")
    test  = pd.read_csv("test.csv")
    df_raw = pd.concat([train, test], ignore_index=True)
    print(f"Success: {len(df_raw):,} records loaded.")
except:
    print("Note: Data files not found in local path. Ensure train.csv and test.csv are available.")

Success: 74,498 records loaded.


## 2 · Preprocessing & Cleaning
Data integrity is paramount. We normalize names, set ordinal orders, and create binary flags for analysis.

In [3]:
df = df_raw.copy()

# Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")

# Create numeric attrition flag for rate calculations
df["attrition_flag"] = (df["attrition"] == "Left").astype(int)

# Define and set Ordinal Category Orders
ORDINALS = {
    "work_life_balance": ["Poor", "Fair", "Good", "Excellent"],
    "job_satisfaction": ["Low", "Medium", "High", "Very High"],
    "performance_rating": ["Low", "Below Average", "Average", "High"],
    "education_level": ["High School", "Associate Degree", "Bachelor’s Degree", "Master’s Degree", "PhD"],
    "job_level": ["Entry", "Mid", "Senior"],
    "company_size": ["Small", "Medium", "Large"],
    "company_reputation": ["Poor", "Fair", "Good", "Excellent"],
    "employee_recognition": ["Low", "Medium", "High", "Very High"]
}

for col, order in ORDINALS.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

# Core calculation helper
def rate_table(data, cols):
    if isinstance(cols, str): cols = [cols]
    out = data.groupby(cols, observed=True)["attrition_flag"].agg(total="count", left="sum").reset_index()
    out["rate"] = (out["left"] / out["total"] * 100).round(1)
    return out

# Insight display helper
def show_answer(q_num, title, insight, action):
    display(Markdown(f"### {q_num} · {title}\n**Insight:** {insight}\n\n**Recommended Action:** {action}"))

company_avg = df["attrition_flag"].mean() * 100
print(f"Preprocessing complete. Global Attrition Rate: {company_avg:.1f}%")

Preprocessing complete. Global Attrition Rate: 47.5%


### 2.1 · Data Quality — Edge Case Detection
Synthetic datasets can contain implausible record combinations. We identify and remove these noisy records before analysis to ensure clean, trustworthy results.

In [4]:
before = len(df)

# Flag implausible combinations
edge_mask = (
    # Very young + advanced degree
    ((df['age'] < 25) & (df['education_level'].isin(["Master's Degree", "PhD"]))) |
    # Entry-level + top-5% salary (data generation artifact)
    ((df['job_level'] == 'Entry') & (df['monthly_income'] > df['monthly_income'].quantile(0.95))) |
    # Less than 1 year + more than 2 promotions (impossible timeline)
    ((df['years_at_company'] <= 1) & (df['number_of_promotions'] > 2))
)

edge_cases = df[edge_mask].copy()
df = df[~edge_mask].copy()

young_phd   = ((edge_cases['age'] < 25) & (edge_cases['education_level'].isin(["Master's Degree","PhD"]))).sum()
entry_pay   = ((edge_cases['job_level'] == 'Entry') & (edge_cases['monthly_income'] > df['monthly_income'].quantile(0.95))).sum()
short_promo = ((edge_cases['years_at_company'] <= 1) & (edge_cases['number_of_promotions'] > 2)).sum()

print(f"Edge cases removed  : {len(edge_cases):,} records ({len(edge_cases)/before*100:.2f}% of dataset)")
print(f"Records retained    : {len(df):,} of {before:,}")
print(f"\nBreakdown:")
print(f"  · Young + advanced degree  : {young_phd:,}")
print(f"  · Entry-level + top-5% pay : {entry_pay:,}")
print(f"  · <1yr tenure + 3+ promos  : {short_promo:,}")

# Recalculate on clean data
company_avg = df['attrition_flag'].mean() * 100
print(f"\nUpdated Global Attrition Rate: {company_avg:.1f}%")

Edge cases removed  : 2,295 records (3.08% of dataset)
Records retained    : 72,203 of 74,498

Breakdown:
  · Young + advanced degree  : 621
  · Entry-level + top-5% pay : 1,512
  · <1yr tenure + 3+ promos  : 185

Updated Global Attrition Rate: 47.3%


## 3 · Exploratory Data Analysis
We now dive into the 10 core questions required by the HR Stakeholders.

### Q1 · The Headline
Understanding the overall scale of attrition and identifying the primary source of leakage.

In [5]:
role_stats = rate_table(df, "job_role").sort_values("left", ascending=False)
top_role = role_stats.iloc[0]

fig = px.bar(role_stats, x="left", y="job_role", orientation="h", text_auto=True,
             title="Employees Who Left by Job Role",
             labels=LABELS, color_discrete_sequence=["#E45756"])
fig.update_layout(template=TEMPLATE, yaxis={'categoryorder':'total ascending'})
fig.show()

show_answer(
    "Q1", "Overall Attrition & Top Leaking Role",
    f"Overall, {company_avg:.1f}% of employees left. The **{top_role['job_role']}** role is the largest contributor to attrition volume, losing {int(top_role['left']):,} people.",
    f"Prioritize **{top_role['job_role']}** for a deep-dive retention review to understand the high absolute leakage."
)

### Q1 · Overall Attrition & Top Leaking Role
**Insight:** Overall, 47.3% of employees left. The **Technology** role is the largest contributor to attrition volume, losing 8,405 people.

**Recommended Action:** Prioritize **Technology** for a deep-dive retention review to understand the high absolute leakage.

### Q2 · Overtime
Investigating if high workload (overtime) is a predictor of attrition.

In [6]:
ot_stats = rate_table(df, "overtime")
ot_yes = ot_stats[ot_stats["overtime"] == "Yes"]["rate"].values[0]
ot_no = ot_stats[ot_stats["overtime"] == "No"]["rate"].values[0]

fig = px.bar(ot_stats, x="overtime", y="rate", color="overtime", text_auto='.1f',
             title="Attrition Rate: Overtime vs No Overtime",
             labels=LABELS, color_discrete_map={"Yes": "#E45756", "No": "#4E79A7"})
fig.add_hline(y=company_avg, line_dash="dash", annotation_text=f"Company Avg ({company_avg:.1f}%)")
fig.update_layout(template=TEMPLATE, showlegend=False)
fig.show()

show_answer(
    "Q2", "Impact of Overtime",
    f"Overtime workers leave at **{ot_yes:.1f}%**, which is significantly higher than non-overtime workers ({ot_no:.1f}%).",
    "HR should audit overtime hours by department and consider capping mandatory overtime to prevent burnout."
)

### Q2 · Impact of Overtime
**Insight:** Overtime workers leave at **51.3%**, which is significantly higher than non-overtime workers (45.4%).

**Recommended Action:** HR should audit overtime hours by department and consider capping mandatory overtime to prevent burnout.

### Q3 · Remote Work
Assessing the retention benefit of remote work while acknowledging its current adoption scale.

In [7]:
remote_stats = rate_table(df, "remote_work")
remote_yes_rate = remote_stats[remote_stats["remote_work"] == "Yes"]["rate"].values[0]
remote_no_rate = remote_stats[remote_stats["remote_work"] == "No"]["rate"].values[0]
remote_pop = (df["remote_work"] == "Yes").mean() * 100

fig = px.bar(remote_stats, x="remote_work", y="rate", color="remote_work", text_auto='.1f',
             title="Attrition Rate: Remote vs On-site",
             labels=LABELS, color_discrete_map={"Yes": "#2f9e66", "No": "#E45756"})
fig.add_hline(y=company_avg, line_dash="dash", annotation_text="Company Avg")
fig.update_layout(template=TEMPLATE, showlegend=False)
fig.show()

show_answer(
    "Q3", "Remote Work Effectiveness",
    f"Remote work appears protective (attrition rate: {remote_yes_rate:.1f}%), but it only applies to {remote_pop:.1f}% of staff.",
    "Pilot expanded remote/hybrid options for high-attrition roles to validate if it reduces turnover at scale."
)

### Q3 · Remote Work Effectiveness
**Insight:** Remote work appears protective (attrition rate: 24.6%), but it only applies to 19.0% of staff.

**Recommended Action:** Pilot expanded remote/hybrid options for high-attrition roles to validate if it reduces turnover at scale.

### Q4 · Pay Fairness
Analyzing attrition within pay bands at the same job level to identify fairness issues.

In [8]:
df['pay_band'] = df.groupby('job_level', observed=True)['monthly_income'].transform(
    lambda x: pd.qcut(x, 4, labels=['Lowest 25%', 'Lower-Mid', 'Upper-Mid', 'Highest 25%'])
)
pay_stats = rate_table(df, ['job_level', 'pay_band'])

fig = px.line(pay_stats, x="pay_band", y="rate", color="job_level", markers=True,
              title="Attrition Rate by Pay Band within Job Levels",
              labels=LABELS)
fig.update_layout(template=TEMPLATE)
fig.show()

show_answer(
    "Q4", "Pay Fairness & Retention",
    "Attrition is highest for the lowest-paid 25% within each job level. The retention benefit of pay flattens after the 'Upper-Mid' band.",
    "Review pay-band midpoints and prioritize raises for those in the 'Lowest 25%' to fix fairness-driven attrition."
)

### Q4 · Pay Fairness & Retention
**Insight:** Attrition is highest for the lowest-paid 25% within each job level. The retention benefit of pay flattens after the 'Upper-Mid' band.

**Recommended Action:** Review pay-band midpoints and prioritize raises for those in the 'Lowest 25%' to fix fairness-driven attrition.

### Q5 · The Retention Timeline
Identifying the 'critical window' where employees are most likely to leave.

In [9]:
df['tenure_stage'] = pd.cut(df['years_at_company'], bins=[-1, 2, 5, 10, 100], 
                            labels=['New (0-2y)', 'Established (3-5y)', 'Experienced (6-10y)', 'Veteran (10y+)'])
tenure_stats = rate_table(df, "tenure_stage")

fig = px.bar(tenure_stats, x="tenure_stage", y="rate", text_auto='.1f',
             title="Attrition Rate by Tenure Stage",
             labels=LABELS, color_discrete_sequence=["#d99120"])
fig.add_hline(y=company_avg, line_dash="dash", annotation_text=f"Company Avg ({company_avg:.1f}%)", 
              annotation_font_color="#555")
fig.update_layout(template=TEMPLATE)
fig.show()

peak_stage = tenure_stats.loc[tenure_stats['rate'].idxmax(), 'tenure_stage']
show_answer(
    "Q5", "The Retention Critical Window",
    f"Attrition peaks in the **{peak_stage}** stage, making the early tenure window the most critical period for retention. "
    "Employees who make it past 5 years show significantly lower exit rates.",
    "Shift retention focus to the first 2 years through structured onboarding, 30/60/90-day check-ins, and first-manager quality assurance."
)

### Q5 · The Retention Critical Window
**Insight:** Attrition peaks in the **New (0-2y)** stage, making the early tenure window the most critical period for retention. Employees who make it past 5 years show significantly lower exit rates.

**Recommended Action:** Shift retention focus to the first 2 years through structured onboarding, 30/60/90-day check-ins, and first-manager quality assurance.

### Q6 · Engagement Warning Signs
Combining satisfaction and work-life balance to create an early-warning matrix.

In [10]:
eng_stats = rate_table(df, ["job_satisfaction", "work_life_balance"])
pivot_eng = eng_stats.pivot(index="job_satisfaction", columns="work_life_balance", values="rate")

fig = px.imshow(pivot_eng, text_auto='.1f', aspect="auto",
                title="Attrition Heatmap: Satisfaction vs Work-Life Balance",
                labels=dict(x="Work-Life Balance", y="Job Satisfaction", color="Rate (%)"),
                color_continuous_scale="RdYlGn_r")
fig.show()

show_answer(
    "Q6", "Engagement Warning Signs",
    "'Low' job satisfaction combined with 'Poor' work-life balance creates a critical flight risk (highest attrition in the matrix).",
    "Managers should treat 'Poor' work-life balance survey results as an immediate trigger for a stay interview."
)

### Q6 · Engagement Warning Signs
**Insight:** 'Low' job satisfaction combined with 'Poor' work-life balance creates a critical flight risk (highest attrition in the matrix).

**Recommended Action:** Managers should treat 'Poor' work-life balance survey results as an immediate trigger for a stay interview.

### Q7 · Life Stage
Exploring how personal demographics influence the decision to stay or leave.

In [11]:
df['age_group'] = pd.cut(df['age'], bins=[17, 30, 45, 100], labels=['Young (18-30)', 'Middle (31-45)', 'Senior (45+)'])
df['dependents_group'] = df['number_of_dependents'].apply(
    lambda x: 'No Dependents' if x == 0 else ('1–2 Dependents' if x <= 2 else '3+ Dependents')
)

# Chart 1: Age × Marital Status
life_stats = rate_table(df, ["age_group", "marital_status"])
fig1 = px.bar(life_stats, x="age_group", y="rate", color="marital_status", barmode="group",
              title="Attrition Rate by Age Group & Marital Status",
              labels=LABELS)
fig1.add_hline(y=company_avg, line_dash="dash", annotation_text=f"Company Avg ({company_avg:.1f}%)")
fig1.update_layout(template=TEMPLATE)
fig1.show()

# Chart 2: Age × Dependents
dep_stats = rate_table(df, ["age_group", "dependents_group"])
fig2 = px.bar(dep_stats, x="age_group", y="rate", color="dependents_group", barmode="group",
              title="Attrition Rate by Age Group & Number of Dependents",
              labels=LABELS)
fig2.add_hline(y=company_avg, line_dash="dash", annotation_text=f"Company Avg ({company_avg:.1f}%)")
fig2.update_layout(template=TEMPLATE)
fig2.show()

# Find highest-risk life-stage combination
full_life = rate_table(df, ["age_group", "marital_status", "dependents_group"])
top_life = full_life.sort_values("rate", ascending=False).iloc[0]

show_answer(
    "Q7", "Life Stage Risk Profiles",
    f"Young (18-30), Single employees with no dependents are at the highest risk — they have the least financial "
    f"and social anchors tying them to the company. The riskiest segment shows {top_life['rate']:.1f}% attrition "
    f"({int(top_life['total']):,} employees). Stability factors — age, marriage, and dependents — all correlate "
    "with significantly higher retention.",
    "Tailor retention offers by life stage: emphasize career growth, mentorship, and skill development for "
    "younger single staff. For employees with families, prioritize flexibility, remote options, and benefits "
    "like childcare support or parental leave to build financial and personal anchors."
)

### Q7 · Life Stage Risk Profiles
**Insight:** Young (18-30), Single employees with no dependents are at the highest risk — they have the least financial and social anchors tying them to the company. The riskiest segment shows 73.7% attrition (2,243 employees). Stability factors — age, marriage, and dependents — all correlate with significantly higher retention.

**Recommended Action:** Tailor retention offers by life stage: emphasize career growth, mentorship, and skill development for younger single staff. For employees with families, prioritize flexibility, remote options, and benefits like childcare support or parental leave to build financial and personal anchors.

### Q8 · Career Stagnation
Testing the hypothesis that a lack of growth opportunities drives employees away.

In [12]:
df['stuck_profile'] = np.where(
    (df['number_of_promotions'] == 0) & 
    (df['leadership_opportunities'] == 'No') & 
    (df['innovation_opportunities'] == 'No'),
    "Stuck (No Growth)", "Growth Access"
)
stuck_stats = rate_table(df, "stuck_profile")

fig = px.bar(stuck_stats, x="stuck_profile", y="rate", color="stuck_profile", text_auto='.1f',
             title="Attrition Rate: Stuck vs Growth Access",
             labels=LABELS, color_discrete_map={"Stuck (No Growth)": "#E45756", "Growth Access": "#2f9e66"})
fig.update_layout(template=TEMPLATE, showlegend=False)
fig.show()

show_answer(
    "Q8", "The Cost of Stagnation",
    f"Employees feeling 'stuck' (no promotions or growth opportunities) leave at a rate of {stuck_stats[stuck_stats['stuck_profile']=='Stuck (No Growth)']['rate'].values[0]:.1f}%.",
    "Introduce an internal mobility program to provide growth even when vertical promotions are unavailable."
)

### Q8 · The Cost of Stagnation
**Insight:** Employees feeling 'stuck' (no promotions or growth opportunities) leave at a rate of 49.8%.

**Recommended Action:** Introduce an internal mobility program to provide growth even when vertical promotions are unavailable.

### Q9 · The Highest-Risk Profile
Synthesizing multiple factors to identify the most vulnerable employee segment.

In [13]:
risk_cols = ["overtime", "job_satisfaction", "work_life_balance"]
profiles = rate_table(df, risk_cols)
top_risk = profiles.sort_values("rate", ascending=False).iloc[0]
lift = top_risk['rate'] - company_avg

show_answer(
    "Q9", "The Highest-Risk Profile",
    f"The highest-risk profile is: **Overtime: {top_risk['overtime']} | Satisfaction: {top_risk['job_satisfaction']} | "
    f"Work-Life Balance: {top_risk['work_life_balance']}**. "
    f"Their attrition rate is **{top_risk['rate']:.1f}%** — {lift:.1f} percentage points above the company average.",
    f"This profile matches **{int(top_risk['total']):,} employees** — a sizeable enough group to act on immediately. "
    "HR should prioritize these individuals for workload audits, manager check-ins, and flexible scheduling."
)

### Q9 · The Highest-Risk Profile
**Insight:** The highest-risk profile is: **Overtime: Yes | Satisfaction: Low | Work-Life Balance: Poor**. Their attrition rate is **71.5%** — 24.2 percentage points above the company average.

**Recommended Action:** This profile matches **309 employees** — a sizeable enough group to act on immediately. HR should prioritize these individuals for workload audits, manager check-ins, and flexible scheduling.

### Q10 · What Moves the Needle
Ranking the top drivers of attrition to prioritize HR interventions for the next quarter.

In [14]:
drivers = [
    ("Overtime", "overtime", "Yes"),
    ("Job Satisfaction", "job_satisfaction", "Low"),
    ("Work-Life Balance", "work_life_balance", "Poor"),
    ("Growth Access", "stuck_profile", "Stuck (No Growth)")
]

driver_results = []
for name, col, val in drivers:
    stats = rate_table(df, col)
    rate = stats[stats[col] == val]["rate"].values[0]
    driver_results.append({"Driver": name, "Value": val, "Rate": rate, "Lift": rate - company_avg})

driver_df = pd.DataFrame(driver_results).sort_values("Lift", ascending=False)
fig = px.bar(driver_df, x="Lift", y="Driver", orientation="h", text_auto='.1f',
             title="Top Drivers of Attrition (Lift Above Baseline)",
             labels={"Lift": "Lift (Percentage Points)"})
fig.update_layout(template=TEMPLATE)
fig.show()

best_driver = driver_df.iloc[0]
show_answer(
    "Q10", "Priority #1 Recommendation",
    f"**{best_driver['Driver']}** is the strongest driver of attrition with a lift of {best_driver['Lift']:.1f} points above average.",
    f"Next quarter, HR should launch a focused initiative to address **{best_driver['Driver']}**, which could prevent hundreds of avoidable exits."
)

### Q10 · Priority #1 Recommendation
**Insight:** **Work-Life Balance** is the strongest driver of attrition with a lift of 12.8 points above average.

**Recommended Action:** Next quarter, HR should launch a focused initiative to address **Work-Life Balance**, which could prevent hundreds of avoidable exits.

## 4 · Executive Summary for Dashboard
Based on the analysis, here are the three core pillars for the HR Dashboard:
1. **Burnout Prevention:** Overtime and Poor Work-Life Balance are the strongest predictors of exit.
2. **Early Intervention:** The first 2 years of tenure are critical; retention efforts must start at onboarding.
3. **Growth as Retention:** Employees with no clear growth signals (promotions/opportunities) are twice as likely to leave.

---
**End of Analysis · Kayfa AI & Data Analytics Internship**